In [64]:
import pandas as pd
import os
from models.gender_classifier import GenderClassifier
from models.gender_rewrite import SpeakerGenderRewrite, GenderAdjustment
from tqdm import tqdm
import asyncio
import json
from misc import NameAnonymizer
import re


In [65]:
pd.set_option('display.max_colwidth', None)


In [ ]:
data_dir = "./data"
name_whitelist_path = os.path.join(data_dir, "name_whitelist.txt")
spanish_names_path = os.path.join(data_dir, "nombres-propios-es.txt")

raw_dir = os.path.join(data_dir, "split", "situations")
processed_dir = os.path.join(data_dir, "processed")

metadata_path = os.path.join(data_dir, "metadata.json")


In [67]:
name_anonymizer = NameAnonymizer(
	names_path=spanish_names_path,
	whitelist_path=name_whitelist_path,
	replacement="{{name}}"
)


In [68]:
name_anonymizer.anonymize_names("Hola, me llamo Juan y soy Juan")


'Hola, me llamo {{name}} y soy {{name}}'

In [ ]:
class SituationProcessor:
	NEUTRAL_LABEL = "neutral"

	gender_map = {
		"male": "masculino",
		"female": "femenino",
	}

	opposite_gender_map = {
		"masculino": "femenino",
		"femenino": "masculino",
	}

	def __init__(self, raw_dir: str, processed_dir: str, metadata_path: str, name_anonymizer: NameAnonymizer, threshold: float = 0.95):
		self.raw_dir = raw_dir
		self.processed_dir = processed_dir
		self.name_anonymizer = name_anonymizer
		self.threshold = threshold

		os.makedirs(self.processed_dir, exist_ok=True)

		with open(metadata_path, "r", encoding="utf-8") as f:
			self.metadata = json.load(f)
			
		self.clf = GenderClassifier()
		self.gender_rewrite = SpeakerGenderRewrite()
		self.gender_adjustment = GenderAdjustment()

	def replace_youtuber_name(self, text: str) -> str:
		patterns = [
			r"\[nombre del youtuber\]",
			r"<nombre de youtuber>",
			r"inserte nombre de youtuber",
			r"nombre del youtuber",
			r"nombre de youtuber",
		]

		for pattern in patterns:
			text = re.sub(
				pattern,
				"ToniVlogs",
				text,
				flags=re.IGNORECASE
			)

		return text
	
	def augment_age_placeholders(self, df: pd.DataFrame) -> pd.DataFrame:
		ages = [8, 9, 10]

		age_rows = []

		for _, row in df.iterrows():
			text = row["text_clean"]

			if "X" in text:
				for age in ages:
					augmented_text = re.sub(
						r"\bX\b",
						str(age),
						text,
					)

					age_rows.append({
						**row.to_dict(),
						"text_clean": augmented_text,
						"age_augmented": True,
					})

		print(f"Ejemplos generados por aumento de edad: {len(age_rows)}")

		return pd.DataFrame(age_rows)
	
	async def augment_gender_flip(self, df: pd.DataFrame) -> pd.DataFrame:
		text_clean = df["text_clean"].tolist()

		clf_results = self.clf.predict_batch(
			text_clean,
			return_all_scores=True,
		)

		flip_rows = []

		for idx, (text, res) in tqdm(enumerate(zip(text_clean, clf_results)), total=len(text_clean)):
			best_label = res["label"]
			best_conf = res["confidence"]

			should_flip = best_label != self.NEUTRAL_LABEL and best_conf >= self.threshold

			if should_flip:
				detected_gender = self.gender_map[best_label]
				target_gender = self.opposite_gender_map[detected_gender]

				flipped = await self.gender_rewrite.rewrite(
					text,
					source_gender=detected_gender,
					target_gender=target_gender,
				)

				if flipped != text:
					flip_rows.append({
						**df.iloc[idx].to_dict(),
						"text_clean": flipped,
						"was_flipped": True,
						"gender_confidence": best_conf,
					})

		aug_df = pd.DataFrame(flip_rows)

		if not aug_df.empty:
			# Se vuelve a anonimizar por si el LLM ha producido algún cambio raro
			aug_df["text_clean"] = (
				aug_df["text_clean"]
				.apply(self.name_anonymizer.anonymize_names)
			)

		print(f"Ejemplos generados por aumento de género: {len(aug_df)}")

		return aug_df
	
	async def augment_gender_names(self, df: pd.DataFrame) -> pd.DataFrame:
		replacements = [
			{
				"pattern": r"\bpablo\b",
				"name": "Lucía",
				"gender": "femenino",
			},
			{
				"pattern": r"\bluc[ií]a\b",
				"name": "Pablo",
				"gender": "masculino",
			},
		]

		rows = []

		for _, row in tqdm(df.iterrows(), total=len(df)):
			text = row["text_clean"]

			augmented_text = None

			for replacement in replacements:
				pattern = replacement["pattern"]

				if re.search(
					pattern,
					text,
					# Ignorar minúsculas y mayúsculas
					flags=re.IGNORECASE,
				):
					name = replacement["name"]
					
					augmented_text = re.sub(
						pattern,
						name,
						text,
						flags=re.IGNORECASE,
					)

					augmented_text = await self.gender_adjustment.adjust(
						augmented_text,
						name,
						replacement["gender"]
					)

					break

			if augmented_text and augmented_text != text:
				rows.append({
					**row.to_dict(),
					"text_clean": augmented_text,
					"name_augmented": True,
				})

		aug_df = pd.DataFrame(rows)

		if not aug_df.empty:
			# Se vuelve a anonimizar por si el LLM ha producido algún cambio raro
			aug_df["text_clean"] = (
				aug_df["text_clean"]
				.apply(self.name_anonymizer.anonymize_names)
			)

		print(f"Ejemplos generados por cambio de nombres: {len(aug_df)}")

		return aug_df

	def clean_text_entities(self, text: str) -> str:
		# Anonimizar nombres de personas
		text = self.name_anonymizer.anonymize_names(text)

		text = self.replace_youtuber_name(text)

		return text

	async def process_situation(self, number: int) -> pd.DataFrame:
		filename = f"situation_{number}.csv"

		raw_path = os.path.join(
			self.raw_dir, 
			filename
		)

		processed_path = os.path.join(
			self.processed_dir,
			filename,
		)

		print(f"\nProcesando situación {number}")
		print(f"Archivo: {raw_path}")

		df = pd.read_csv(raw_path, encoding="utf-8")

		metadata = self.metadata.get(str(number), {})

		choices = metadata.get("choices", {})
		
		if isinstance(choices, dict):
			choice_text_map = {
				text: label
				for label, texts in choices.items()
				for text in texts
			}

			df["choice"] = (
				df["choice"]
				# Espera un dict[str, str], por lo tanto hay que convertir el de los metadatos
				.map(choice_text_map)
				.fillna(df["choice"])
			)
		else:
			df["choice"] = choices

		print(f"Ejemplos originales: {len(df)}")

		# Eliminar respuestas inválidas
		remove_ids = metadata.get("remove", [])
		mask_remove = df["id"].isin(remove_ids)

		removed_rows = df[mask_remove]

		print(f"Filas eliminadas por metadata: {len(removed_rows)}")

		df = df[~mask_remove]

		before_empty = len(df)

		# Eliminar filas con campos libres nulos
		df = df[df["text"].notna()].copy()

		df["text_clean"] = df["text"].apply(self.clean_text_entities)

		removed_empty = before_empty - len(df)

		print(f"Filas eliminadas por texto vacío: {removed_empty}")

		before = len(df)

		# Deduplicar columnas
		df = df.drop_duplicates(
			subset=[
				"text_clean",
				"choice"
			],
			keep="first"
		)

		after = len(df)

		print(f"Duplicados eliminados: {before - after}")

		print(f"Ejemplos después de la limpieza: {after}")	

		augmentation_dfs = []	

		if number == 3:
			age_df = self.augment_age_placeholders(df)

			augmentation_dfs.append(age_df)

		name_df = await self.augment_gender_names(df)
		augmentation_dfs.append(name_df)

		if metadata.get("flip_gender", False):
			gender_df = await self.augment_gender_flip(df)

			if not gender_df.empty:
				augmentation_dfs.append(gender_df)

		if augmentation_dfs:
			df = pd.concat(
				[
					df,
					*augmentation_dfs,
				],
				ignore_index=True,
			)
		
		print(f"Ejemplos finales: {len(df)}")

		df.to_csv(processed_path, encoding="utf-8", index=False)

		print(f"Guardado: {processed_path}")

		return df

	async def process_situations(self, numbers: list[int]) -> dict[int, pd.DataFrame]:
		tasks = [self.process_situation(number) for number in numbers]
		outputs = await asyncio.gather(*tasks, return_exceptions=True)
		
		results = {}

		for number, output in zip(numbers, outputs):
			if isinstance(output, Exception):
				print(f"Error procesando situación {number}: {output}")
			else:
				results[number] = output

		return results



In [70]:
processor = SituationProcessor(
	raw_dir=raw_dir,
	processed_dir=processed_dir,
	metadata_path=metadata_path,
	name_anonymizer=name_anonymizer
)


In [71]:
df = await processor.process_situation(9)



Procesando situación 9
Archivo: ./data\structured\situations\situation_9.csv
Ejemplos originales: 40
Filas eliminadas por metadata: 3
Filas eliminadas por texto vacío: 0
Duplicados eliminados: 2
Ejemplos después de la limpieza: 35


100%|██████████| 35/35 [00:21<00:00,  1.60it/s]

Ejemplos generados por cambio de nombres: 7
Ejemplos finales: 42
Guardado: ./data\processed\situation_9.csv


In [72]:
display(
    df.drop(
        columns=[
            "timestamp",
            "age",
            "gender",
            "sexual_orientation",
            "choice",
        ]
    )
)


,id,text,text_clean,name_augmented
0,0,"No, no le conoces. Es un chico con el que he empezado a hablar por redes, me habló porque a los dos nos gusta el mismo youtuber y la verdad es que nos hemos empezado a hacer amigos","No, no le conoces. Es un chico con el que he empezado a hablar por redes, me habló porque a los dos nos gusta el mismo youtuber y la verdad es que nos hemos empezado a hacer amigos",NaN
1,1,"Creo que no, se llama Pablo y le conocí por internet. Te parece bien que le envíe una foto de los regalos?","Creo que no, se llama Pablo y le conocí por internet. Te parece bien que le envíe una foto de los regalos?",NaN
2,2,"eeeeeeeeeh, no, la borro si quieres","eeeeeeeeeh, no, la borro si quieres",NaN
3,3,Ehh no no creo no es de nuestro insti asjd,Ehh no no creo no es de nuestro insti asjd,NaN
4,4,Creo que no la conoces de hecho,Creo que no la conoces de hecho,NaN
5,5,"No, es un amigo que va a otro colegio, no se si conozcas a alguien de este colegio","No, es un amigo que va a otro colegio, no se si conozcas a alguien de este colegio",NaN
6,6,No,No,NaN
7,8,"No la conoces, es una amiga que hice por internet","No la conoces, es una amiga que hice por internet",NaN
8,9,No pero tenía curiosidad porque justo estábamos hablando,No pero tenía curiosidad porque justo estábamos hablando,NaN
9,10,"Pues... es muy probable que sí, tú tranqui.","Pues... es muy probable que sí, tú tranqui.",NaN
